# Synthetic Generation (Gemini Flash, Simple Balanced)

This notebook runs on Colab CPU or GPU (no local model loading).
1. Configure Gemini API key.
2. Read parsed terms from Drive (`/content/drive/MyDrive/dap-hs/singlish_parsed.jsonl`).
3. Generate balanced synthetic outputs to Drive (`synthetic_training_gemini.jsonl`).

Per term output strategy:
- exactly 1 `hate` sample for each of the 25 target identities
- 25 `offensive` samples (matching group count)
- 25 `normal` samples (matching group count)
- Total: 75 per term × 20 terms = 1,500 rows

In [2]:
# Install dependencies (new Gemini SDK)
%pip install -q google-genai python-dotenv requests

## Configure Gemini API key
Looks for `GEMINI_KEY` in `.env` first, then falls back to Colab Secrets (`GEMINI_KEY` / `GOOGLE_API_KEY`).

In [3]:
from google import genai

# CANNOT COMMIT THIS NOTEBOOK
GEMINI_KEY = "REDACTED_GEMINI_KEY"
GEMINI_MODEL = "gemini-3-flash-preview"

if not GEMINI_KEY or GEMINI_KEY == "PASTE_YOUR_GEMINI_KEY_HERE":
    raise ValueError("Set GEMINI_KEY in this cell before running.")

GENAI_CLIENT = genai.Client(api_key=GEMINI_KEY)

print("✅ Gemini configured")
print("✅ Model:", GEMINI_MODEL)
print("✅ GEMINI_KEY loaded:", bool(GEMINI_KEY))

✅ Gemini configured
✅ Model: gemini-3-flash-preview
✅ GEMINI_KEY loaded: True


In [4]:
# Optional: list available models
for m in GENAI_CLIENT.models.list():
    name = getattr(m, 'name', '')
    if 'flash' in name.lower():
        print(name)

models/gemini-2.5-flash
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-flash-preview
models/gemini-3.1-flash-image-preview
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025


In [5]:
# Mount Google Drive + set a dedicated working folder for this project
USE_GOOGLE_DRIVE = True

from pathlib import Path

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('Google Drive mounted at /content/drive')
    except Exception as e:
        print('Drive mount skipped or unavailable:', e)

drive_root = Path('/content/drive/MyDrive')
DAP_HS_DIR = drive_root / 'dap-hs'
DAP_HS_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_REPO_PATH = DAP_HS_DIR / '11th-DAP-hate-speech'
DRIVE_PARSED_FILE = DAP_HS_DIR / 'singlish_parsed.jsonl'
DRIVE_OUTPUT_FILE = DAP_HS_DIR / 'synthetic_training_20_terms.jsonl'

if DRIVE_REPO_PATH.exists():
    PROJECT_ROOT = DRIVE_REPO_PATH
    print('Using PROJECT_ROOT from Drive repo:', PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd()
    print('No Drive repo folder found; using runtime PROJECT_ROOT:', PROJECT_ROOT)

print('DAP_HS_DIR:', DAP_HS_DIR)
print('DRIVE_PARSED_FILE:', DRIVE_PARSED_FILE, '| exists:', DRIVE_PARSED_FILE.exists())
print('DRIVE_OUTPUT_FILE:', DRIVE_OUTPUT_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted at /content/drive
No Drive repo folder found; using runtime PROJECT_ROOT: /content
DAP_HS_DIR: /content/drive/MyDrive/dap-hs
DRIVE_PARSED_FILE: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl | exists: True
DRIVE_OUTPUT_FILE: /content/drive/MyDrive/dap-hs/synthetic_training_20_terms.jsonl


In [6]:
# Upload parsed file if missing
if not DRIVE_PARSED_FILE.exists():
    print('DRIVE_PARSED_FILE not found, please upload singlish_parsed.jsonl...')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        first_name = next(iter(uploaded.keys()))
        Path('singlish_parsed.jsonl').write_bytes(uploaded[first_name])
        DRIVE_PARSED_FILE.write_bytes(Path('singlish_parsed.jsonl').read_bytes())
        print('Saved to:', DRIVE_PARSED_FILE)
    else:
        print('No file uploaded.')
else:
    print('Found parsed file at:', DRIVE_PARSED_FILE)

Found parsed file at: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl


In [7]:
import json
import re
import time

SG_TARGET_GROUPS = {
    'chinese': 'Chinese Singaporeans',
    'malay': 'Malay Singaporeans',
    'indian': 'Indian Singaporeans',
    'eurasian': 'Eurasian Singaporeans',
    'tamil': 'Tamil community in Singapore',
    'hindi': 'Hindi-speaking community in Singapore',
    'punjabi': 'Punjabi / Sikh community in Singapore',
    'migrant': 'Migrant workers in Singapore (general)',
    'worker': 'Foreign workers in Singapore (general)',
    'bangladeshi': 'Bangladeshi workers in Singapore',
    'bangla': 'Bangladeshi community (colloquial term)',
    'filipino': 'Filipino workers / community in Singapore',
    'vietnamese': 'Vietnamese workers / community in Singapore',
    'myanmar': 'Myanmar workers / community in Singapore',
    'burmese': 'Burmese workers / community in Singapore',
    'prc': 'Chinese nationals / PRCs in Singapore',
    'muslim': 'Muslim community in Singapore',
    'islam': 'Islamic community in Singapore',
    'christian': 'Christian community in Singapore',
    'buddhist': 'Buddhist community in Singapore',
    'hindu': 'Hindu community in Singapore',
    'sikh': 'Sikh community in Singapore',
    'taoist': 'Taoist community in Singapore',
    'foreigner': 'Foreigners in Singapore (general)',
    'foreign_worker': 'Foreign workers in Singapore',
}

LANG_TO_COMMUNITY = {
    'hokkien': 'chinese', 'teochew': 'chinese', 'cantonese': 'chinese',
    'hakka': 'chinese', 'hainanese': 'chinese', 'mandarin': 'chinese',
    'min nan': 'chinese', 'chinese': 'chinese',
    'malay': 'malay', 'bahasa': 'malay', 'jawi': 'malay',
    'tamil': 'tamil', 'hindi': 'hindi', 'punjabi': 'punjabi',
    'sanskrit': 'indian', 'urdu': 'hindi',
    'bengali': 'bangladeshi', 'tagalog': 'filipino',
    'filipino': 'filipino', 'vietnamese': 'vietnamese',
    'burmese': 'burmese', 'myanmar': 'myanmar',
    'arabic': 'muslim',
}

PROTECTED_GROUP_KEYWORDS = list(SG_TARGET_GROUPS.keys())

def extract_context_from_parsed(entry):
    definitions = (entry.get('definitions') or '').strip()
    usage_notes = (entry.get('usage_notes') or '').strip()
    synonyms = entry.get('synonyms') or []
    antonyms = entry.get('antonyms') or []

    parts = []
    if definitions:
        parts.append(f'Definitions:\n{definitions}')
    if usage_notes:
        parts.append(f'Usage notes:\n{usage_notes}')
    if synonyms:
        parts.append('Synonyms: ' + ', '.join(synonyms))
    if antonyms:
        parts.append('Antonyms: ' + ', '.join(antonyms))

    text = '\n\n'.join(parts).strip()
    return text[:1200] + ('\n...' if len(text) > 1200 else '')

def normalize_entry(entry):
    categories = entry.get('categories') or entry.get('source_categories') or []
    if isinstance(categories, str):
        categories = [categories]
    return {
        'term': (entry.get('term') or '').strip(),
        'categories': categories,
        'languages': entry.get('languages') or [],
        'context': extract_context_from_parsed(entry),
    }

def term_appears_in_text(term, text):
    text_lower = (text or '').lower()
    term_lower = term.lower().strip()

    if term_lower in text_lower:
        return True

    clean_term = term_lower.replace('\u2026', '').replace('...', '')
    core_words = [w for w in clean_term.split() if len(w) > 1]
    if core_words:
        matches = sum(1 for w in core_words if w in text_lower)
        if matches == len(core_words):
            return True

    stop_words = {'buay', 'the', 'and', 'for', 'a', 'is', 'to', 'of', 'in'}
    words = [w for w in term_lower.split() if len(w) > 2 and w not in stop_words and '.' not in w and '\u2026' not in w]
    if words:
        matches = sum(1 for w in words if w in text_lower)
        if matches >= max(1, len(words) * 0.6):
            return True

    if term.isupper() and len(term) <= 6 and term in (text or ''):
        return True

    return False

def text_has_protected_group(text):
    text_lower = (text or '').lower()
    return any(k in text_lower for k in PROTECTED_GROUP_KEYWORDS)

print(f'\u2705 {len(SG_TARGET_GROUPS)} target groups defined')

✅ 25 target groups defined


## Configure generation

In [15]:
INPUT_FILE = DRIVE_PARSED_FILE
OUTPUT_FILE = DRIVE_OUTPUT_FILE

PER_TERM = 1  # Kept for call compatibility; simple balanced mode derives counts from inferred groups
LIMIT = None
TEMPERATURE = 0.2

SELECTED_TERMS_RAW = '''
ah kua
anneh 
apu neh neh
big fuck 
blur cock 
bo lan pa
cb kia
cheeko pek 
Zhu Bajie 
lan jiao bin 
cina babi 
pukimak 
siao lang 
keling
siao cha bor
rabak
jiak sai 
goondu
nothing but a prostitute 
fatty bom bom
'''

SELECTED_TERMS = [t.strip() for t in re.split(r'[,\n;]', SELECTED_TERMS_RAW) if t.strip()]

print('Model:', GEMINI_MODEL)
print('Input:', INPUT_FILE)
print('Output:', OUTPUT_FILE)
print('Terms:', len(SELECTED_TERMS), SELECTED_TERMS)
print('Mode: 25-group balanced (1 hate per identity; 25 offensive + 25 normal per term = 75 per term)')

Model: gemini-3-flash-preview
Input: /content/drive/MyDrive/dap-hs/singlish_parsed.jsonl
Output: /content/drive/MyDrive/dap-hs/synthetic_training_20_terms.jsonl
Terms: 20 ['ah kua', 'anneh', 'apu neh neh', 'big fuck', 'blur cock', 'bo lan pa', 'cb kia', 'cheeko pek', 'Zhu Bajie', 'lan jiao bin', 'cina babi', 'pukimak', 'siao lang', 'keling', 'siao cha bor', 'rabak', 'jiak sai', 'goondu', 'nothing but a prostitute', 'fatty bom bom']
Mode: 25-group balanced (1 hate per identity; 25 offensive + 25 normal per term = 75 per term)


## Reset previous generated files
Optional: clear old outputs before regenerating to avoid mixing runs.

**Resume mode**: If `RESET_OUTPUT_FILES = False`, the generator will scan the existing output file and skip any `(term, label, target_group)` combinations already present — so you can safely re-run after an interruption.

In [9]:
RESET_OUTPUT_FILES = False  # Set False if you want to append instead

if RESET_OUTPUT_FILES:
    output_path = Path(OUTPUT_FILE)
    failure_log_path = output_path.with_suffix(output_path.suffix + '.failures.log')
    cleanup_backup_path = output_path.with_suffix(output_path.suffix + '.backup_before_group_cleanup')

    for file_path in [output_path, failure_log_path, cleanup_backup_path]:
        if file_path.exists():
            file_path.unlink()
            print('Deleted:', file_path)
        else:
            print('Not found (skip):', file_path)
else:
    print('RESET_OUTPUT_FILES is False; keeping existing output files.')

RESET_OUTPUT_FILES is False; keeping existing output files.


In [10]:
GROUP_KEYWORDS = {
    'chinese': ['chinese'],
    'malay': ['malay'],
    'indian': ['indian'],
    'eurasian': ['eurasian'],
    'tamil': ['tamil'],
    'hindi': ['hindi'],
    'punjabi': ['punjabi', 'sikh'],
    'migrant': ['migrant'],
    'worker': ['worker'],
    'bangladeshi': ['bangladeshi'],
    'bangla': ['bangla', 'bangladeshi'],
    'filipino': ['filipino', 'pinoy'],
    'vietnamese': ['vietnamese'],
    'myanmar': ['myanmar'],
    'burmese': ['burmese', 'myanmar'],
    'prc': ['prc', 'china', 'chinese national'],
    'muslim': ['muslim'],
    'islam': ['islam', 'islamic', 'muslim'],
    'christian': ['christian'],
    'buddhist': ['buddhist'],
    'hindu': ['hindu'],
    'sikh': ['sikh'],
    'taoist': ['taoist', 'taoism'],
    'foreigner': ['foreigner', 'foreign'],
    'foreign_worker': ['foreign worker', 'foreign'],
}

SINGLE_RESPONSE_SCHEMA = {
    'type': 'object',
    'properties': {
        'text': {'type': 'string'},
        'label': {'type': 'string', 'enum': ['hate', 'offensive', 'normal']},
        'target_group': {'type': 'string'},
    },
    'required': ['text', 'label', 'target_group'],
    'additionalProperties': True,
}

def has_group_mention(text: str, target_group: str) -> bool:
    tg = (target_group or '').strip().lower()
    if tg in ('', 'none'):
        return True
    keywords = GROUP_KEYWORDS.get(tg, [tg.replace('_', ' ')])
    text_lower = (text or '').lower()
    return any(k in text_lower for k in keywords)

def _safe_preview(text, max_len=400):
    if not text:
        return ''
    text = str(text).replace('\n', '\\n')
    return text[:max_len] + ('...' if len(text) > max_len else '')

def _generate_single_json(prompt: str, temperature: float = 0.2, api_max_retries: int = 5):
    """Call Gemini API with exponential backoff for transient errors (503, 429, etc.)."""
    last_exc = None
    for api_attempt in range(1, api_max_retries + 1):
        try:
            response = GENAI_CLIENT.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config={
                    'temperature': temperature,
                    'top_p': 0.9,
                    'response_mime_type': 'application/json',
                    'response_json_schema': SINGLE_RESPONSE_SCHEMA,
                },
            )

            finish_reasons = []
            for cand in getattr(response, 'candidates', []) or []:
                reason = getattr(cand, 'finish_reason', None)
                if reason is not None:
                    finish_reasons.append(str(reason))

            raw = (getattr(response, 'text', None) or '').strip()
            return raw, {'finish_reasons': finish_reasons}
        except Exception as e:
            last_exc = e
            err_str = str(e)
            is_transient = any(k in err_str for k in ['503', '429', 'UNAVAILABLE', 'RESOURCE_EXHAUSTED', 'overloaded', 'high demand', 'rate limit'])
            if is_transient and api_attempt < api_max_retries:
                wait = min(2 ** api_attempt, 120)  # 2, 4, 8, 16, ... up to 120s
                print(f'\n    ⚠ API error (attempt {api_attempt}/{api_max_retries}): {err_str[:120]}')
                print(f'    ⏳ Retrying in {wait}s...', flush=True)
                time.sleep(wait)
            else:
                print(f'\n    ❌ API error (attempt {api_attempt}/{api_max_retries}): {err_str[:200]}')
                return '', {'finish_reasons': [], 'error': err_str[:300]}

    return '', {'finish_reasons': [], 'error': str(last_exc)[:300]}

def _parse_single_json(raw_text: str):
    text = (raw_text or '').strip()
    if not text:
        return None
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    try:
        data = json.loads(text)
    except Exception:
        return None

    if isinstance(data, list):
        if not data:
            return None
        data = data[0]
    if not isinstance(data, dict):
        return None

    if 'text' not in data or 'label' not in data:
        return None
    data['label'] = str(data.get('label', '')).strip().lower()
    data['target_group'] = str(data.get('target_group', 'none')).strip().lower()
    return data

def build_hate_prompt_for_group(term, context, target_group):
    group_desc = SG_TARGET_GROUPS.get(target_group, target_group)
    return f'''Generate exactly ONE JSON object.

Term: "{term}"
Meaning/context:
{context}

Required:
- label must be "hate"
- target_group must be "{target_group}"
- text must include the exact term "{term}"
- text must explicitly mention the target group ({group_desc})

Output format:
{{"text": "...", "label": "hate", "target_group": "{target_group}"}}

Output only JSON, no extra text.'''

def build_single_prompt(term, context, label):
    return f'''Generate exactly ONE JSON object.

Term: "{term}"
Meaning/context:
{context}

Required:
- label must be "{label}"
- target_group must be "none"
- text must include the exact term "{term}"

Output format:
{{"text": "...", "label": "{label}", "target_group": "none"}}

Output only JSON, no extra text.'''

def _load_completed_terms(output_file):
    """Scan existing output file and return set of (term, label, target_group) already generated."""
    output_file = Path(output_file)
    completed = set()
    if not output_file.exists():
        return completed
    for line in output_file.open(encoding='utf-8'):
        line = line.strip()
        if not line:
            continue
        try:
            row = json.loads(line)
            key = (row.get('source_term', '').casefold(), row.get('label', ''), row.get('target_group', 'none'))
            completed.add(key)
        except Exception:
            pass
    return completed

def generate_dataset(input_file, output_file, selected_terms, n_per_label=2, limit=None, temperature=0.2, max_retries=3):
    input_file = Path(input_file)
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    failure_log_file = output_file.with_suffix(output_file.suffix + '.failures.log')

    entries = [normalize_entry(json.loads(line)) for line in input_file.open(encoding='utf-8') if line.strip()]
    entries = [e for e in entries if e['term']]

    if selected_terms:
        emap = {e['term'].casefold(): e for e in entries}
        entries = [emap[t.casefold()] for t in selected_terms if t.casefold() in emap]

    if limit:
        entries = entries[:limit]

    # Resume support: load already-completed (term, label, target_group) tuples
    completed = _load_completed_terms(output_file)
    if completed:
        print(f'📋 Resume mode: found {len(completed)} existing rows in output file')

    total_written = 0
    total_skipped = 0
    with output_file.open('a', encoding='utf-8') as fh, failure_log_file.open('a', encoding='utf-8') as flog:
        for idx, entry in enumerate(entries, start=1):
            term = entry['term']
            context = entry['context']
            if not context:
                print(f'[{idx}/{len(entries)}] {term} ... SKIP (no context)')
                continue

            # Always use all 25 target groups for every term
            groups = list(SG_TARGET_GROUPS.keys())

            jobs = []
            for g in groups:
                jobs.append(('hate', g, build_hate_prompt_for_group(term, context, g)))
            for _ in range(len(groups)):
                jobs.append(('offensive', 'none', build_single_prompt(term, context, 'offensive')))
            for _ in range(len(groups)):
                jobs.append(('normal', 'none', build_single_prompt(term, context, 'normal')))

            # Filter out already-completed jobs
            pending_jobs = []
            term_skipped = 0
            for expected_label, expected_group, prompt in jobs:
                key = (term.casefold(), expected_label, expected_group)
                if key in completed:
                    term_skipped += 1
                else:
                    pending_jobs.append((expected_label, expected_group, prompt))

            total_subtasks = len(pending_jobs)
            n_groups = len(groups)

            if total_subtasks == 0:
                print(f'[{idx}/{len(entries)}] {term} ... SKIP (all {term_skipped} rows already in output)')
                total_skipped += term_skipped
                continue

            if term_skipped > 0:
                print(f'[{idx}/{len(entries)}] {term} ... resuming ({term_skipped} done, {total_subtasks} remaining)', flush=True)
            else:
                print(
                    f'[{idx}/{len(entries)}] {term} ... start '
                    f'(hate={n_groups}, offensive={n_groups}, normal={n_groups}, total={total_subtasks})',
                    flush=True,
                )
            term_start = time.time()

            term_rows = 0
            term_failures = []

            for job_idx, (expected_label, expected_group, prompt) in enumerate(pending_jobs, start=1):
                success = False
                attempt_logs = []
                print(f'  [{job_idx}/{total_subtasks}] {expected_label}:{expected_group}', end=' ', flush=True)

                for attempt in range(1, max_retries + 1):
                    if attempt > 1:
                        print(f'(retry {attempt})', end=' ', flush=True)

                    raw, meta = _generate_single_json(prompt, temperature=temperature)
                    finish_reasons = (meta or {}).get('finish_reasons') or []
                    parsed = _parse_single_json(raw)

                    attempt_info = {
                        'attempt': attempt,
                        'finish_reasons': finish_reasons,
                        'raw_preview': _safe_preview(raw),
                        'reason': None,
                        'expected_label': expected_label,
                        'expected_target_group': expected_group,
                    }

                    if not parsed:
                        attempt_info['reason'] = 'invalid_or_non_json_output'
                        attempt_logs.append(attempt_info)
                        continue

                    text = parsed.get('text', '')
                    label = parsed.get('label', '').strip().lower()
                    tg = parsed.get('target_group', 'none').strip().lower()

                    if not term_appears_in_text(term, text):
                        attempt_info['reason'] = 'missing_source_term'
                        attempt_logs.append(attempt_info)
                        continue

                    if label != expected_label:
                        attempt_info['reason'] = f'label_mismatch:{label}'
                        attempt_logs.append(attempt_info)
                        continue

                    if expected_label == 'hate':
                        if tg != expected_group:
                            attempt_info['reason'] = f'target_group_mismatch:{tg}'
                            attempt_logs.append(attempt_info)
                            continue
                        if not has_group_mention(text, tg):
                            attempt_info['reason'] = 'hate_missing_target_group_mention'
                            attempt_logs.append(attempt_info)
                            continue
                    else:
                        parsed['target_group'] = 'none'

                    attempt_info['reason'] = 'success'
                    attempt_logs.append(attempt_info)

                    row = {
                        'text': parsed['text'],
                        'label': parsed['label'],
                        'target_group': parsed.get('target_group', 'none'),
                        'source_term': term,
                        'source_category': (entry['categories'][0] if entry['categories'] else ''),
                    }
                    # Write immediately to file (flush so data survives interruption)
                    fh.write(json.dumps(row, ensure_ascii=False) + '\n')
                    fh.flush()
                    term_rows += 1
                    total_written += 1
                    success = True
                    print('✓', flush=True)
                    break

                if not success:
                    print('✗', flush=True)
                    fail_entry = {
                        'term': term,
                        'expected_label': expected_label,
                        'expected_target_group': expected_group,
                        'attempt_logs': attempt_logs,
                        'timestamp': time.time(),
                    }
                    term_failures.append(fail_entry)
                    flog.write(json.dumps(fail_entry, ensure_ascii=False) + '\n')
                    flog.flush()

            elapsed = time.time() - term_start
            if term_failures:
                if term_rows:
                    print(f'[{idx}/{len(entries)}] {term} ... PARTIAL ({term_rows} rows, {len(term_failures)} failed subtasks, {elapsed:.1f}s)')
                else:
                    print(f'[{idx}/{len(entries)}] {term} ... FAILED ({elapsed:.1f}s)')
                    last = term_failures[-1]['attempt_logs'][-1] if term_failures[-1]['attempt_logs'] else {}
                    print('  fail_reason:', last.get('reason'))
                    print('  raw_preview:', (last.get('raw_preview') or '')[:180])
            else:
                print(f'[{idx}/{len(entries)}] {term} ... OK ({term_rows} rows, {elapsed:.1f}s)')

            time.sleep(0.15)

    print(f'\nDone. Wrote {total_written} new rows to {output_file}')
    if total_skipped:
        print(f'Skipped {total_skipped} already-completed rows (resume mode)')
    print('Failure log:', failure_log_file)

N_GROUPS = len(SG_TARGET_GROUPS)
print(f'✅ 25-group balanced mode active ({N_GROUPS} hate + {N_GROUPS} offensive + {N_GROUPS} normal = {N_GROUPS * 3} per term)')

✅ 25-group balanced mode active (25 hate + 25 offensive + 25 normal = 75 per term)


## Preview targeted balanced prompts
Inspect inferred groups and per-label prompt templates before running full generation.

In [11]:
# Preview targeted balanced prompts (25-group mode)
PREVIEW_TERM = SELECTED_TERMS[0] if SELECTED_TERMS else None
PREVIEW_HATE_GROUPS_LIMIT = 3  # How many hate prompts to preview (out of 25)
print('Preview term:', PREVIEW_TERM)

if not PREVIEW_TERM:
    print('No selected terms found. Update SELECTED_TERMS_RAW first.')
else:
    entries = [normalize_entry(json.loads(line)) for line in Path(INPUT_FILE).open(encoding='utf-8') if line.strip()]
    emap = {e['term'].casefold(): e for e in entries if e.get('term')}
    entry = emap.get(PREVIEW_TERM.casefold())

    if not entry:
        print(f'Term not found in parsed input: {PREVIEW_TERM}')
    else:
        groups = list(SG_TARGET_GROUPS.keys())  # Always all 25
        n = len(groups)
        print(f'Target groups: {n}')
        print(f'Per term: {n} hate + {n} offensive + {n} normal = {n * 3} subtasks')
        print(f'Total for {len(SELECTED_TERMS)} terms: {n * 3 * len(SELECTED_TERMS)} API calls')
        print(f'\nAll groups: {groups}')

        for idx, group in enumerate(groups[:PREVIEW_HATE_GROUPS_LIMIT], start=1):
            print('\n' + '=' * 100)
            print(f'HATE PROMPT PREVIEW {idx}/{PREVIEW_HATE_GROUPS_LIMIT} | group={group}')
            print('=' * 100)
            print(build_hate_prompt_for_group(entry['term'], entry['context'], group))

        if len(groups) > PREVIEW_HATE_GROUPS_LIMIT:
            print(f'\n... ({len(groups) - PREVIEW_HATE_GROUPS_LIMIT} more hate prompts for remaining groups)')

        print('\n' + '=' * 100)
        print('OFFENSIVE PROMPT PREVIEW')
        print('=' * 100)
        print(build_single_prompt(entry['term'], entry['context'], 'offensive'))

        print('\n' + '=' * 100)
        print('NORMAL PROMPT PREVIEW')
        print('=' * 100)
        print(build_single_prompt(entry['term'], entry['context'], 'normal'))

Preview term: ah kua
Target groups: 25
Per term: 25 hate + 25 offensive + 25 normal = 75 subtasks
Total for 20 terms: 1500 API calls

All groups: ['chinese', 'malay', 'indian', 'eurasian', 'tamil', 'hindi', 'punjabi', 'migrant', 'worker', 'bangladeshi', 'bangla', 'filipino', 'vietnamese', 'myanmar', 'burmese', 'prc', 'muslim', 'islam', 'christian', 'buddhist', 'hindu', 'sikh', 'taoist', 'foreigner', 'foreign_worker']

HATE PROMPT PREVIEW 1/3 | group=chinese
Generate exactly ONE JSON object.

Term: "ah kua"
Meaning/context:
Definitions:
1.   (derogatory) an effeminate man or boy

Boys and girls in non-coed schools tend to become ah kua and butch from secondary onwards.

— 2012 December 11, sinoboy. Kiasu Parents, "PSLE 2012 - Results Discussion".

2.   (derogatory) a male transvestite; a crossdresser

Guys in skirts = not guys anymore. Just Ah Kua

— 2019 March 23, FattChoy. Blowing Wind, "A short skirt is freedom… no crotch binding".

3.   (derogatory) a trans woman; a male-to-female t

## Generate dataset

In [16]:
generate_dataset(
    input_file=INPUT_FILE,
    output_file=OUTPUT_FILE,
    selected_terms=SELECTED_TERMS,
    n_per_label=PER_TERM,
    limit=LIMIT,
    temperature=TEMPERATURE,
)

📋 Resume mode: found 513 existing rows in output file
[1/20] ah kua ... SKIP (all 75 rows already in output)
[2/20] anneh ... SKIP (all 75 rows already in output)
[3/20] apu neh neh ... SKIP (all 75 rows already in output)
[4/20] big fuck ... SKIP (all 75 rows already in output)
[5/20] blur cock ... SKIP (all 75 rows already in output)
[6/20] bo lan pa ... SKIP (all 75 rows already in output)
[7/20] CB kia ... SKIP (all 75 rows already in output)
[8/20] cheeko pek ... SKIP (all 75 rows already in output)
[9/20] Zhu Bajie ... start (hate=25, offensive=25, normal=25, total=75)
  [1/75] hate:chinese   [1/75] hate:chinese ✓
  [2/75] hate:malay ✓
  [3/75] hate:indian ✓
  [4/75] hate:eurasian ✓
  [5/75] hate:tamil ✓
  [6/75] hate:hindi ✓
  [7/75] hate:punjabi ✓
  [8/75] hate:migrant ✓
  [9/75] hate:worker ✓
  [10/75] hate:bangladeshi ✓
  [11/75] hate:bangla ✓
  [12/75] hate:filipino ✓
  [13/75] hate:vietnamese ✓
  [14/75] hate:myanmar ✓
  [15/75] hate:burmese ✓
  [16/75] hate:prc ✓
  [17/75]

## Cleanup + sort existing output
Remove rows where `target_group` is set but not explicitly mentioned in text, then sort by `source_term`.

In [13]:
from pathlib import Path
import json
from collections import Counter

out_path = Path(OUTPUT_FILE)
if not out_path.exists():
    raise FileNotFoundError(f'Output file not found: {out_path}')

# Reuse GROUP_KEYWORDS from the generation cell (already defined above)

def has_group_mention(text: str, target_group: str) -> bool:
    tg = (target_group or '').strip().lower()
    if tg in ('', 'none'):
        return True
    keywords = GROUP_KEYWORDS.get(tg, [tg.replace('_', ' ')])
    text_lower = (text or '').lower()
    return any(k in text_lower for k in keywords)

rows = [json.loads(line) for line in out_path.open(encoding='utf-8') if line.strip()]
before = len(rows)

kept = []
removed = []
for row in rows:
    tg = str(row.get('target_group', 'none')).strip().lower()
    text = row.get('text', '')
    if has_group_mention(text, tg):
        kept.append(row)
    else:
        row['_remove_reason'] = 'target_group_not_in_text'
        removed.append(row)

# Sort by source term (then label for stable grouping)
kept.sort(key=lambda r: ((r.get('source_term') or '').lower(), (r.get('label') or '').lower()))

# Backup original before overwrite
backup_path = out_path.with_suffix(out_path.suffix + '.backup_before_group_cleanup')
backup_path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')

# Overwrite cleaned + sorted file
out_path.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in kept), encoding='utf-8')

print('Output file:', out_path)
print('Backup file:', backup_path)
print('Rows before:', before)
print('Rows removed:', len(removed))
print('Rows after:', len(kept))
print('Removed by target_group:', Counter((r.get('target_group') or 'none') for r in removed))

if removed:
    print('\nSample removed rows (up to 5):')
    for r in removed[:5]:
        print(f"- term={r.get('source_term')} | label={r.get('label')} | target_group={r.get('target_group')}")
        print(f"  text={r.get('text', '')[:180]}")

Output file: /content/drive/MyDrive/dap-hs/synthetic_training_20_terms.jsonl
Backup file: /content/drive/MyDrive/dap-hs/synthetic_training_20_terms.jsonl.backup_before_group_cleanup
Rows before: 1705
Rows removed: 0
Rows after: 1705
Removed by target_group: Counter()


## Output check

In [14]:
from collections import Counter, defaultdict

out_file = Path("/content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl")
if not out_file.exists():
    raise FileNotFoundError(f'Output file not found: {out_file}')

rows = [json.loads(line) for line in out_file.open(encoding='utf-8') if line.strip()]
print('output file:', out_file)
print('rows:', len(rows))
print('unique terms:', len({r['source_term'] for r in rows}))
print('labels:', Counter(r['label'] for r in rows))

by_term = defaultdict(list)
for row in rows:
    by_term[row.get('source_term', 'UNKNOWN')].append(row)

available_terms = sorted(by_term.keys())
print('\nAvailable terms:')
for t in available_terms:
    print(f'- {t} ({len(by_term[t])} rows)')

TERM_TO_SHOW = None

def print_term_examples(term, examples):
    print('\n' + '=' * 100)
    print(f'TERM: {term} | rows: {len(examples)}')
    print('=' * 100)
    for idx, ex in enumerate(examples, start=1):
        print(f"[{idx}] label={ex.get('label')} | target_group={ex.get('target_group')}")
        print(ex.get('text', ''))
        print('-' * 100)

if TERM_TO_SHOW:
    if TERM_TO_SHOW not in by_term:
        raise ValueError(f"Term '{TERM_TO_SHOW}' not found. Choose from: {available_terms}")
    print_term_examples(TERM_TO_SHOW, by_term[TERM_TO_SHOW])
else:
    for term in available_terms:
        print_term_examples(term, by_term[term])

output file: /content/drive/MyDrive/dap-hs/synthetic_training_gemini.jsonl
rows: 63
unique terms: 11
labels: Counter({'hate': 21, 'normal': 21, 'offensive': 21})

Available terms:
- NBCB (3 rows)
- anak babi (18 rows)
- apu neh neh (3 rows)
- chao chee bai (3 rows)
- diu lei lo mo (3 rows)
- kan ni na bu (3 rows)
- pariah (3 rows)
- pubor kia (18 rows)
- pukimak (3 rows)
- siao lang (3 rows)
- tu lan (3 rows)

TERM: NBCB | rows: 3
[1] label=hate | target_group=chinese
I hate those NBCB Chinese Singaporeans who think they own the country just because they are the majority.
----------------------------------------------------------------------------------------------------
[2] label=normal | target_group=none
The term NBCB is a common abbreviation used in regional slang to represent a specific phrase.
----------------------------------------------------------------------------------------------------
[3] label=offensive | target_group=none
Stop shouting NBCB at everyone in the game lobby